# Bake activation-event metadata into COGs on S3 (bulk)

Point this at **one input: an `s3://bucket/prefix`**. For every `.tif`/`.TIF` object under it whose
name starts with a valid `YYYYMM_Hazard_Location_...` it will:

1. **Bake** the activation event into the GeoTIFF tags (`ACTIVATION_EVENT`, plus parsed
   `YEAR_MONTH` / `HAZARD` / `LOCATION`, and `PROCESSOR`).
2. **Rename in place** — write a new object under the **same prefix/folder** with the
   `YYYYMM_Hazard_Location_` prefix stripped out of the filename (the event now lives in the
   metadata, not the name), then **delete the old key**.
3. Keep the COG otherwise **identical** — same CRS, geotransform, pixels, nodata, dtype, band
   count. Compression is lossless, so pixel values are bit-for-bit unchanged.

**Idempotent** — a file that already carries an `ACTIVATION_EVENT` tag is **skipped** (re-run any
time). The skip is enforced authoritatively at write time, from the object's own tags.

**Can't determine the event?** Objects whose name has no valid `YYYYMM_Hazard_Location` are written
to a local **`needs_event.csv`**. Fill in the `ACTIVATION_EVENT` column and run the **Bulk-fix**
section at the bottom to bake those too (their key is left unchanged — there is no prefix to strip).

> **Run with `DRY_RUN = True` (the default) first.** The top cells only *report* the plan — nothing
> is uploaded or deleted — so you can review before committing. Set `DRY_RUN = False` and re-run.

> **Why re-encode instead of editing in place?** GDAL 3.10+ refuses to add tags to an existing COG
> in place, so the object is rewritten with its own (lossless) compression — data stays identical.
> COG validity is checked locally with `rio_cogeo.cog_validate` (via `validate_cog_in_memory`), the
> same check the VEDA tiler uses. Runs on the Disasters JupyterHub with the ambient `disasters-prod`
> identity (read+write to `nasa-disasters`, same account) — no assume-role needed.

In [ ]:
# =====================================================================
# CONFIG  --  the only thing you need to edit
# =====================================================================
INPUT = "s3://nasa-disasters/drcs_activations_new/202510_Flood_AK/"   # <-- s3://bucket/prefix

DRY_RUN    = True            # True = report only (no uploads, no deletes). Set False to run.
SOURCE     = "TBD"           # optional SOURCE tag (e.g. USGS, Copernicus, CSDA); baked into every file
AWS_REGION = "us-west-2"
CSV_PATH   = "needs_event.csv"   # local CSV of files needing an event (edit it for the bulk-fix)

# --- make `shared_utils` importable when running from the notebooks/ folder ---
import sys, os
from pathlib import Path
_root = Path.cwd()
for _ in range(6):
    if (_root / "src" / "shared_utils").is_dir() or (_root / "shared_utils").is_dir():
        break
    _root = _root.parent
for _p in (str(_root), str(_root / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print("INPUT    :", INPUT)
print("DRY_RUN  :", DRY_RUN, "  (True = report only)")
print("SOURCE   :", SOURCE)
print("CSV_PATH :", CSV_PATH)

In [ ]:
import re, csv
from collections import Counter
from urllib.parse import urlparse
import boto3, rasterio
from rasterio.io import MemoryFile

from shared_utils.cog_metadata import (
    create_cog_with_metadata,
    validate_cog_in_memory,
    resolve_metadata,
    DEFAULT_ACTIVATION_PATTERN,   # ^(\d{6})_([A-Za-z0-9]+)_([A-Za-z0-9]+)_(.*)\.tif$
)
from shared_utils.version import PROCESSOR_STRING
from shared_utils.s3_operations import list_s3_files, setup_vsi_credentials
from shared_utils.parallel import map_threaded

_ACT_RE   = re.compile(DEFAULT_ACTIVATION_PATTERN, re.IGNORECASE)   # group 4 = remainder to keep
_EVENT_RE = re.compile(r'^\d{6}_[A-Za-z0-9]+_[A-Za-z0-9]+')          # a valid YYYYMM_Hazard_Location

def parse_s3(uri):
    u = urlparse(uri)
    return u.netloc, u.path.lstrip("/")

BUCKET, PREFIX = parse_s3(INPUT)
s3 = boto3.client("s3", region_name=AWS_REGION)   # ambient disasters-prod on the hub; no assume-role
setup_vsi_credentials(s3)                          # let GDAL /vsis3 read object headers with these creds


def read_event_s3(bucket, key):
    # Return the object's existing ACTIVATION_EVENT tag, or None.
    # Fast path: /vsis3 header range-read (no full download). Fallback: download bytes
    # (used when /vsis3 is unavailable, e.g. offline tests). A successful header open that
    # simply lacks the tag returns None WITHOUT downloading.
    try:
        with rasterio.open("/vsis3/{}/{}".format(bucket, key)) as s_:
            return s_.tags().get("ACTIVATION_EVENT")
    except Exception:
        pass
    try:
        b = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
        with MemoryFile(b) as mf, mf.open() as s_:
            return s_.tags().get("ACTIVATION_EVENT")
    except Exception:
        return None


def new_key_for(key):
    # Strip a leading YYYYMM_Hazard_Location_ from the BASENAME; keep the folder; normalize ext to .tif.
    # Returns (new_key, event) or (None, None) if the basename has no valid event prefix.
    folder, base = os.path.dirname(key), os.path.basename(key)
    m = _ACT_RE.match(base)
    if not (m and m.group(4)):
        return None, None
    event = "{}_{}_{}".format(m.group(1), m.group(2), m.group(3))
    new_base = m.group(4) + ".tif"
    return (folder + "/" + new_base if folder else new_base), event


# --- scan ---
keys = list_s3_files(s3, BUCKET, PREFIX, suffix=".tif")
already_baked, matched, unmatched = [], [], []
for k in keys:
    ev = read_event_s3(BUCKET, k)
    if ev and _EVENT_RE.match(ev):
        already_baked.append({"bucket": BUCKET, "key": k, "event": ev})
        continue
    nk, event = new_key_for(k)
    if nk:
        matched.append({"bucket": BUCKET, "key": k, "event": event, "new_key": nk})
    else:
        unmatched.append({"bucket": BUCKET, "key": k, "reason": "no_event_pattern"})

# collision guard: two sources -> same new_key, or new_key already exists as a different object
_existing = set(keys)
_counts = Counter(m["new_key"] for m in matched)
_kept = []
for m in matched:
    dup = _counts[m["new_key"]] > 1
    exists_other = m["new_key"] in _existing and m["new_key"] != m["key"]
    if dup or exists_other:
        unmatched.append({"bucket": BUCKET, "key": m["key"], "reason": "name_collision"})
    else:
        _kept.append(m)
matched = _kept

print("Scanned {} .tif object(s) under s3://{}/{}".format(len(keys), BUCKET, PREFIX))
print("  already baked (skip) : {}".format(len(already_baked)))
print("  to rename + bake     : {}".format(len(matched)))
print("  need an event (CSV)  : {}".format(len(unmatched)))

In [ ]:
# ---- Plan report (safe; writes nothing to S3, only the local CSV) ----
if matched:
    print("WILL RENAME + BAKE (in place, old key deleted):")
    for m in matched:
        print("  {}".format(m["key"]))
        print("      -> {}    [ACTIVATION_EVENT={}]".format(m["new_key"], m["event"]))
else:
    print("Nothing to rename+bake.")

if already_baked:
    print("\nALREADY BAKED (skipped): {}".format(len(already_baked)))
    for a in already_baked[:20]:
        print("  {}    [ACTIVATION_EVENT={}]".format(a["key"], a["event"]))
    if len(already_baked) > 20:
        print("  ... and {} more".format(len(already_baked) - 20))

# Export objects we cannot auto-process to a LOCAL CSV so they can be fixed in bulk below.
# Preserve any ACTIVATION_EVENT already filled in a prior run (re-scanning never wipes your edits).
_prev = {}
if os.path.exists(CSV_PATH):
    with open(CSV_PATH, newline="") as f:
        for r in csv.DictReader(f):
            ev = (r.get("ACTIVATION_EVENT") or "").strip()
            if ev:
                _prev[(r.get("bucket", ""), r.get("key", ""))] = ev
with open(CSV_PATH, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["bucket", "key", "filename", "reason", "ACTIVATION_EVENT"])
    for u in unmatched:
        w.writerow([u["bucket"], u["key"], os.path.basename(u["key"]), u["reason"],
                    _prev.get((u["bucket"], u["key"]), "")])

if unmatched:
    print("\nNEED AN EVENT NAME ({}) -> {}".format(len(unmatched), os.path.abspath(CSV_PATH)))
    for u in unmatched:
        print("  {}    ({})".format(u["key"], u["reason"]))
    print("\n  Edit that CSV (fill the ACTIVATION_EVENT column), then run the Bulk-fix section below.")
else:
    print("\nNo unprocessable objects. (empty {} written)".format(CSV_PATH))

In [ ]:
def _verify_bytes(out_bytes, src_bytes, expected_event):
    # Structural, fast: valid COG + every data property matches source + event tag present.
    reasons = []
    ok, info = validate_cog_in_memory(out_bytes, "out.tif")
    if not ok:
        reasons.append("output is not a valid COG: {}".format(info.get("errors")))
    with MemoryFile(src_bytes) as ma, ma.open() as a, MemoryFile(out_bytes) as mb, mb.open() as b:
        if a.crs != b.crs:                              reasons.append("CRS changed ({} -> {})".format(a.crs, b.crs))
        if (a.width, a.height) != (b.width, b.height):  reasons.append("dimensions changed")
        if a.count != b.count:                          reasons.append("band count changed")
        if a.dtypes != b.dtypes:                        reasons.append("dtype changed")
        if a.nodatavals != b.nodatavals:                reasons.append("nodata changed ({} -> {})".format(a.nodatavals, b.nodatavals))
        if a.transform != b.transform:                  reasons.append("geotransform changed")
        if b.tags().get("ACTIVATION_EVENT") != expected_event:
            reasons.append("ACTIVATION_EVENT tag missing/wrong")
    return (not reasons), reasons


def bake_s3(bucket, key, event, new_key):
    # Download bytes -> (idempotency check from tags) -> re-encode with event baked in, COG kept
    # identical -> validate -> upload new_key -> confirm it landed -> delete old key (only if the
    # key actually changed). new_key == key performs an in-place backfill (used by the bulk-fix).
    try:
        src = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
        with MemoryFile(src) as mf, mf.open() as s_:
            existing = s_.tags().get("ACTIVATION_EVENT")
            bx = int(s_.profile.get("blockxsize") or 512)
            by = int(s_.profile.get("blockysize") or 512)
            try:
                n_ov = len(s_.overviews(1))
            except Exception:
                n_ov = 0
        if existing and _EVENT_RE.match(existing):
            return {"key": key, "status": "skipped", "reason": "already baked ({})".format(existing)}

        meta = resolve_metadata(
            os.path.basename(new_key), mode="manual",
            manual_metadata={"ACTIVATION_EVENT": event, "SOURCE": SOURCE, "PROCESSOR": PROCESSOR_STRING},
        )
        out = create_cog_with_metadata(
            src, meta, output_path=None,
            preserve_compression=True, target_crs=None, web_optimized=False,
            blockxsize=bx, blockysize=by, overview_level=(n_ov if n_ov > 0 else 5),
            quiet=True,
        )
        out = bytes(out)   # engine returns a bytearray; MemoryFile / put_object want bytes
        ok, reasons = _verify_bytes(out, src, event)
        if not ok:
            return {"key": key, "status": "failed", "reasons": reasons}

        s3.put_object(Bucket=bucket, Key=new_key, Body=out, ContentType="image/tiff")
        s3.head_object(Bucket=bucket, Key=new_key)          # confirm the new object landed...
        if new_key != key:
            s3.delete_object(Bucket=bucket, Key=key)        # ...only then remove the original
        return {"key": key, "new_key": new_key, "event": event, "status": "success"}
    except Exception as e:
        return {"key": key, "status": "failed", "reasons": [repr(e)]}


# ---- Execute (guarded by DRY_RUN) ----
if DRY_RUN:
    results = []
    print("DRY_RUN is True -- nothing uploaded or deleted. {} object(s) would be processed.".format(len(matched)))
    print("Set DRY_RUN = False in the CONFIG cell and re-run the cells to execute.")
elif not matched:
    results = []
    print("No objects to process.")
else:
    results = map_threaded(lambda m: bake_s3(m["bucket"], m["key"], m["event"], m["new_key"]),
                           matched, max_workers=4, desc="Baking")
    ok = sum(1 for r in results if r.get("status") == "success")
    sk = sum(1 for r in results if r.get("status") == "skipped")
    print("\nDone: {} succeeded, {} skipped, {} failed.".format(ok, sk, len(results) - ok - sk))
    for r in results:
        if r.get("status") == "failed":
            print("  FAILED", r["key"], "->", r.get("reasons"))
        elif r.get("status") == "skipped":
            print("  SKIP  ", r["key"], "->", r.get("reason"))

---
## Bulk-fix: objects that had no `YYYYMM_Hazard_Location`

The scan above wrote **`needs_event.csv`** (local), one row per object it couldn't auto-process.
Open it, fill in the **`ACTIVATION_EVENT`** column (format `YYYYMM_Hazard_Location`, e.g.
`202510_Flood_AK`) for the rows you want to fix, save it, then run the cell below.

For these objects the event is baked into the **metadata only** — the S3 key is left unchanged
(there was no prefix to strip). The same guarantees apply: the COG is kept identical, an object
already baked is skipped, and `DRY_RUN` must be `False` to actually write.

In [ ]:
with open(CSV_PATH, newline="") as f:
    rows = list(csv.DictReader(f))

fix_plan, fix_skip = [], []
for row in rows:
    bucket = (row.get("bucket") or "").strip()
    key = (row.get("key") or "").strip()
    event = (row.get("ACTIVATION_EVENT") or "").strip()
    if not (bucket and key):
        continue
    if not event:
        fix_skip.append((key, "no ACTIVATION_EVENT provided")); continue
    if not _EVENT_RE.match(event):
        fix_skip.append((key, "invalid event '{}' (need YYYYMM_Hazard_Location)".format(event))); continue
    fix_plan.append((bucket, key, event))

print("Bulk-fix plan: {} to bake, {} skipped".format(len(fix_plan), len(fix_skip)))
for key, why in fix_skip:
    print("  SKIP {}: {}".format(key, why))
for bucket, key, event in fix_plan:
    print("  BAKE {}  [ACTIVATION_EVENT={}]  (key unchanged)".format(key, event))

if DRY_RUN:
    fix_results = []
    print("\nDRY_RUN is True -- nothing written. Set DRY_RUN = False and re-run to bake these.")
elif not fix_plan:
    fix_results = []
    print("\nNothing to bake (fill the ACTIVATION_EVENT column in the CSV first).")
else:
    fix_results = map_threaded(lambda t: bake_s3(t[0], t[1], t[2], t[1]),  # in place: new_key == key
                               fix_plan, max_workers=4, desc="Bulk-fix")
    ok = sum(1 for r in fix_results if r.get("status") == "success")
    sk = sum(1 for r in fix_results if r.get("status") == "skipped")
    print("\nDone: {} succeeded, {} skipped, {} failed.".format(ok, sk, len(fix_results) - ok - sk))
    for r in fix_results:
        if r.get("status") == "failed":
            print("  FAILED", r["key"], "->", r.get("reasons"))
        elif r.get("status") == "skipped":
            print("  SKIP  ", r["key"], "->", r.get("reason"))